In [4]:
import pandas as pd

tracks = pd.read_csv('../data/fma_metadata/tracks.csv', index_col=0, header=[0, 1])

print("Total tracks:", len(tracks))
print("\nColumns available:", tracks.columns.tolist()[:10])
print("\nGenre distribution:")
print(tracks['track']['genre_top'].value_counts())
print("\nSample album titles:")
print(tracks['album']['title'].head(10))
print("\nSample artist names:")
print(tracks['artist']['name'].head(10))
print("\nHow many have non-empty album tags:")
print((tracks['album']['tags'] != '[]').sum())

Total tracks: 106574

Columns available: [('album', 'comments'), ('album', 'date_created'), ('album', 'date_released'), ('album', 'engineer'), ('album', 'favorites'), ('album', 'id'), ('album', 'information'), ('album', 'listens'), ('album', 'producer'), ('album', 'tags')]

Genre distribution:
genre_top
Rock                   14182
Experimental           10608
Electronic              9372
Hip-Hop                 3552
Folk                    2803
Pop                     2332
Instrumental            2079
International           1389
Classical               1230
Jazz                     571
Old-Time / Historic      554
Spoken                   423
Country                  194
Soul-RnB                 175
Blues                    110
Easy Listening            24
Name: count, dtype: int64

Sample album titles:
track_id
2      AWOL - A Way Of Life
3      AWOL - A Way Of Life
5      AWOL - A Way Of Life
10        Constant Hitmaker
20                    Niris
26                    Niris
30    

In [5]:
# Filter to fma_small tracks only
import os

# Get list of track IDs actually present in fma_small
fma_small_ids = []
for folder in os.listdir('../data/fma_small'):
    folder_path = os.path.join('../data/fma_small', folder)
    if os.path.isdir(folder_path):
        for f in os.listdir(folder_path):
            if f.endswith('.mp3'):
                track_id = int(f.replace('.mp3', ''))
                fma_small_ids.append(track_id)

print(f"Total MP3s found in fma_small: {len(fma_small_ids)}")

# Filter metadata to only fma_small tracks
small_tracks = tracks[tracks.index.isin(fma_small_ids)].copy()
print(f"Matched metadata rows: {len(small_tracks)}")

# Genre distribution for small subset
print("\nGenre distribution (fma_small only):")
print(small_tracks['track']['genre_top'].value_counts())

# How many have artist + album title (needed for MusicBrainz lookup)
has_artist = small_tracks['artist']['name'].notna()
has_album  = small_tracks['album']['title'].notna()
print(f"\nTracks with both artist + album title: {(has_artist & has_album).sum()}")

# Preview clean pairs
print("\nSample (artist, album) pairs:")
print(small_tracks[['artist', 'album']][['artist', 'album']].head(10).to_string())

Total MP3s found in fma_small: 0
Matched metadata rows: 0

Genre distribution (fma_small only):
Series([], Name: count, dtype: int64)

Tracks with both artist + album title: 0

Sample (artist, album) pairs:
Empty DataFrame
Columns: [(artist, active_year_begin), (artist, active_year_end), (artist, associated_labels), (artist, bio), (artist, comments), (artist, date_created), (artist, favorites), (artist, id), (artist, latitude), (artist, location), (artist, longitude), (artist, members), (artist, name), (artist, related_projects), (artist, tags), (artist, website), (artist, wikipedia_page), (album, comments), (album, date_created), (album, date_released), (album, engineer), (album, favorites), (album, id), (album, information), (album, listens), (album, producer), (album, tags), (album, title), (album, tracks), (album, type)]
Index: []


In [7]:
# Cell 3 — Diagnose fma_small structure
import os

base = '../data/fma_small'

print("Top level contents:")
print(os.listdir(base))

print("\nFirst subfolder contents (first 5 files):")
first_sub = os.listdir(base)[0]
print(f"Folder: {first_sub}")
print(os.listdir(os.path.join(base, first_sub))[:5])

Top level contents:
['135', '132', '104', '103', '150', '102', '105', '133', '134', '151', '024', '023', '015', '.DS_Store', '012', '079', '046', '041', '048', '077', '083', '084', '070', '013', '014', '022', '025', '071', '085', '049', '082', '076', '040', '078', '047', '065', '091', '096', '062', '054', '053', '098', '038', '007', '000', '009', '036', '031', '052', '099', '055', '063', '097', '090', '064', '030', '008', '037', '001', '039', '006', '145', '142', '129', '116', '111', '118', '127', '120', '143', '144', '121', '119', '126', '110', '128', '117', '153', '154', '131', '136', '109', '100', '107', '138', '155', 'checksums', '152', '106', '139', '101', '137', '108', '130', '089', '042', '045', '087', '073', '074', '080', '020', '027', '018', '011', '016', '029', '081', '075', '072', '086', '044', '088', '043', '017', '028', '010', '026', '019', '021', 'README.txt', '003', '004', '032', '035', '095', '061', '066', '092', '059', '050', '057', '068', '034', '033', '005', '002', '

In [8]:
# Cell 4 — Fixed MP3 scanner
import os

base = '../data/fma_small'
skip = {'.DS_Store', 'checksums', 'README.txt'}

fma_small_ids = []

for folder in os.listdir(base):
    if folder in skip:
        continue
    folder_path = os.path.join(base, folder)
    if not os.path.isdir(folder_path):
        continue
    for f in os.listdir(folder_path):
        if f.endswith('.mp3'):
            try:
                track_id = int(f.replace('.mp3', ''))
                fma_small_ids.append(track_id)
            except:
                pass

print(f"Total MP3s found: {len(fma_small_ids)}")
print(f"Sample track IDs: {sorted(fma_small_ids)[:10]}")

# Filter metadata to fma_small tracks
small_tracks = tracks[tracks.index.isin(fma_small_ids)].copy()
print(f"Matched metadata rows: {len(small_tracks)}")

print("\nGenre distribution (fma_small only):")
print(small_tracks['track']['genre_top'].value_counts())

print(f"\nTracks with artist + album name:")
has_artist = small_tracks['artist']['name'].notna()
has_album  = small_tracks['album']['title'].notna()
print(f"  Both present: {(has_artist & has_album).sum()}")

print("\nSample (artist, album) pairs:")
sample = small_tracks[['artist', 'album']].head(5)
for idx, row in sample.iterrows():
    print(f"  ID {idx}: '{row['artist']['name']}' — '{row['album']['title']}'")

Total MP3s found: 8000
Sample track IDs: [2, 5, 10, 140, 141, 148, 182, 190, 193, 194]
Matched metadata rows: 8000

Genre distribution (fma_small only):
genre_top
Hip-Hop          1000
Pop              1000
Folk             1000
Experimental     1000
Rock             1000
International    1000
Electronic       1000
Instrumental     1000
Name: count, dtype: int64

Tracks with artist + album name:
  Both present: 8000

Sample (artist, album) pairs:
  ID 2: 'AWOL' — 'AWOL - A Way Of Life'
  ID 5: 'AWOL' — 'AWOL - A Way Of Life'
  ID 10: 'Kurt Vile' — 'Constant Hitmaker'
  ID 140: 'Alec K. Redfearn & the Eyesores' — 'The Blind Spot'
  ID 141: 'Alec K. Redfearn & the Eyesores' — 'Every Man For Himself'


In [11]:
# Cell 5 — Fetch cover art from MusicBrainz + Cover Art Archive
import os, time, requests, json
import musicbrainzngs
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm

musicbrainzngs.set_useragent("SpectroGen", "1.0", "gowdaamithaj@gmail.com")

# Output dirs
os.makedirs('../data/covers', exist_ok=True)
os.makedirs('../data/logs',   exist_ok=True)

PROGRESS_FILE = '../data/logs/fetch_progress.json'
FAILED_FILE   = '../data/logs/failed.json'

# Load existing progress (resume if interrupted)
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        progress = json.load(f)  # {track_id: mbid}
else:
    progress = {}

if os.path.exists(FAILED_FILE):
    with open(FAILED_FILE) as f:
        failed = json.load(f)    # [track_id, ...]
else:
    failed = []

print(f"Already fetched: {len(progress)} | Failed: {len(failed)}")

def search_mbid(artist, album):
    try:
        result = musicbrainzngs.search_releases(
            artist=artist, release=album, limit=1
        )
        if result['release-list']:
            return result['release-list'][0]['id']
    except Exception as e:
        pass
    return None

def fetch_cover(mbid, track_id, size=250):
    url = f"https://coverartarchive.org/release/{mbid}/front-{size}"
    try:
        r = requests.get(url, allow_redirects=True, timeout=10)
        if r.status_code == 200:
            img = Image.open(BytesIO(r.content)).convert('RGB')
            img = img.resize((64, 64), Image.LANCZOS)
            img.save(f'../data/covers/{track_id}.jpg')
            return True
    except Exception as e:
        pass
    return False

# Main fetch loop
already_done = set(str(k) for k in progress.keys())
failed_set   = set(str(k) for k in failed)
todo = small_tracks[
    ~small_tracks.index.astype(str).isin(already_done | failed_set)
]

print(f"Tracks remaining to process: {len(todo)}")

for track_id, row in tqdm(todo.iterrows(), total=len(todo)):
    artist = row['artist']['name']
    album  = row['album']['title']

    # Step 1: find MBID
    mbid = search_mbid(artist, album)
    time.sleep(1)  # MusicBrainz rate limit

    if not mbid:
        failed.append(str(track_id))
        with open(FAILED_FILE, 'w') as f:
            json.dump(failed, f)
        continue

    # Step 2: fetch cover art
    success = fetch_cover(mbid, track_id)
    time.sleep(0.5)

    if success:
        progress[str(track_id)] = mbid
        with open(PROGRESS_FILE, 'w') as f:
            json.dump(progress, f)
    else:
        failed.append(str(track_id))
        with open(FAILED_FILE, 'w') as f:
            json.dump(failed, f)

print(f"\nDone!")
print(f"Successfully fetched: {len(progress)} cover images")
print(f"Failed: {len(failed)}")
print(f"Success rate: {len(progress)/8000*100:.1f}%")

Already fetched: 0 | Failed: 0
Tracks remaining to process: 8000


  0%|          | 0/8000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [10]:
!pip install librosa musicbrainzngs requests pandas numpy Pillow tqdm


  Using cached pillow-12.2.0-cp310-cp310-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached scipy-1.15.3-cp310-cp310-macosx_14_0_arm64.whl.metadata (61 kB)
  Using cached scikit_learn-1.7.2-cp310-cp310-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached lazy_loader-0.5-py3-none-any.whl.metadata (5.9 kB)
  Using cached msgpack-1.1.2-cp310-cp310-macosx_11_0_arm64.whl.metadata (8.1 kB)
  Using cached llvmlite-0.47.0-cp310-cp310-macosx_11_0_arm64.whl.metadata (5.0 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached pillow-12.2.0-cp310-cp310-macosx_11_0_arm64.whl (4.7 MB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached lazy_loader-0.5-py3-none-any.whl (8.0 kB)
Using cached msgpack-1.1.2-cp310-cp310-macosx_11_0_arm64.whl (83 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os, time, requests, json
from concurrent.futures import ThreadPoolExecutor, as_completed
import musicbrainzngs
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm

musicbrainzngs.set_useragent("SpectroGen", "1.0", "gowdaamithaj@gmail.com")

os.makedirs('../data/covers', exist_ok=True)
os.makedirs('../data/logs',   exist_ok=True)

PROGRESS_FILE = '../data/logs/fetch_progress.json'
FAILED_FILE   = '../data/logs/failed.json'

# Load existing progress
progress = json.load(open(PROGRESS_FILE)) if os.path.exists(PROGRESS_FILE) else {}
failed   = json.load(open(FAILED_FILE))   if os.path.exists(FAILED_FILE)   else []

print(f"Already fetched: {len(progress)} | Failed: {len(failed)}")

def search_mbid(artist, album):
    try:
        result = musicbrainzngs.search_releases(
            artist=artist, release=album, limit=1
        )
        if result['release-list']:
            return result['release-list'][0]['id']
    except:
        pass
    return None

def fetch_cover(mbid, track_id):
    url = f"https://coverartarchive.org/release/{mbid}/front-250"
    try:
        r = requests.get(url, allow_redirects=True, timeout=10)
        if r.status_code == 200:
            img = Image.open(BytesIO(r.content)).convert('RGB')
            img = img.resize((64, 64), Image.LANCZOS)
            img.save(f'../data/covers/{track_id}.jpg')
            return True
    except:
        pass
    return False

def process_track(args):
    track_id, artist, album = args
    mbid = search_mbid(artist, album)
    time.sleep(0.5)
    if not mbid:
        return track_id, None, False
    success = fetch_cover(mbid, track_id)
    time.sleep(0.3)
    return track_id, mbid, success

# Build todo list
done_set   = set(progress.keys())
failed_set = set(str(k) for k in failed)
todo = [
    (str(idx), row['artist']['name'], row['album']['title'])
    for idx, row in small_tracks.iterrows()
    if str(idx) not in done_set and str(idx) not in failed_set
]

print(f"Tracks remaining: {len(todo)}")
print("Starting parallel fetch with 5 workers...")

# Parallel execution with 5 workers
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(process_track, args): args for args in todo}
    
    for future in tqdm(as_completed(futures), total=len(todo)):
        track_id, mbid, success = future.result()
        
        if success:
            progress[track_id] = mbid
        else:
            if track_id not in failed:
                failed.append(track_id)
        
        # Save progress every 50 tracks
        if (len(progress) + len(failed)) % 50 == 0:
            with open(PROGRESS_FILE, 'w') as f:
                json.dump(progress, f)
            with open(FAILED_FILE, 'w') as f:
                json.dump(failed, f)

# Final save
with open(PROGRESS_FILE, 'w') as f: json.dump(progress, f)
with open(FAILED_FILE,   'w') as f: json.dump(failed, f)

print(f"\nDone!")
print(f"Successfully fetched: {len(progress)} cover images")
print(f"Failed:               {len(failed)}")
print(f"Success rate:         {len(progress)/8000*100:.1f}%")

Already fetched: 59 | Failed: 77
Tracks remaining: 7864
Starting parallel fetch with 5 workers...


  0%|          | 0/7864 [00:00<?, ?it/s]